In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
sample_df = spark.read.option("multiLine", True).json("/Volumes/pyspark/stream/sparkstreaming/sparksource")

In [0]:
dbutils.fs.mkdirs("/Volumes/pyspark/stream/sparkstreaming/sparkarchive")

Infer Schema doesn't work in Spark Streaming, why not work? cause there is very high chance that json structure get change in future so bug has surity,  so need to define schema for json. here is easy way - 

In [0]:
aircraft_schema = StructType([
    StructField("hex", StringType(), True),
    StructField("flight", StringType(), True),
    StructField("t", StringType(), True),
    StructField("alt_baro", IntegerType(), True),
    StructField("gs", DoubleType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("squawk", StringType(), True)
])

This Schema Is Excellent
You covered the core telemetry triad:

Identity → hex
Position → lat, lon
Motion → gs, alt_baro
Time → now

That’s enough for:

✅ Tracking
✅ Aggregation
✅ Windowing
✅ Anomaly detection
✅ Dashboards

In [0]:
my_schema = StructType([
    StructField("ac", ArrayType(aircraft_schema), True),
    StructField("now", DoubleType(), True)
])

In [0]:
source_path = "dbfs:/Volumes/pyspark/stream/sparkstreaming/sparksource"
archive_path = "dbfs:/Volumes/pyspark/stream/sparkstreaming/sparkarchive"
checkpoint_path = "dbfs:/Volumes/pyspark/stream/sparkstreaming/checkpoints"
output_path = "dbfs:/Volumes/pyspark/stream/sparkstreaming/output_table"

# 4. READ the stream
raw_stream_df = spark.readStream.format('json') \
    .option("multiLine", True) \
    .schema(my_schema) \
    .option("cleanSource", "archive") \
    .option("sourceArchiveDir", archive_path) \
    .load(source_path)

%md
In PySpark, df.withColumn("items", **explode_outer**("items")) is a transformation used to deal with **Arrays** or Maps.

- explode(): If the array is null or empty, the entire row is deleted from your results.
- explode_outer(): If the array is null or empty, the row is kept, but the new column will contain a null value.


Example - 

[
  {
    "university": "Stanford",
    "departments": [
      {
        "dept_name": "Physics",
        "courses": ["Quantum", "Mechanics"]
      },
      {
        "dept_name": "History",
        "courses": []
      }
    ]
  },
  {
    "university": "MIT",
    "departments": null
  }
]


first we explode the each list to create rows then flattened it.

''' df_depts = df.withColumn("dept", F.explode_outer("departments"))
df_final = df_depts.withColumn("course_name", F.explode_outer("dept.courses"))
df_final = df_final.select(
    "university",
    F.col("dept.dept_name").alias("department"),
    "course_name"
)'''



In [0]:
flattened_df = raw_stream_df.select(
    explode(col("ac")).alias("aircraft"), 
    col("now").alias("api_timestamp")
).select(
    "aircraft.*", 
    "api_timestamp"
).withColumn("processing_time", current_timestamp())

%md
Delta is strongly recommended for streaming.

Delta Lake is a storage format that combines:
    Parquet files + Transaction Log = Delta Table
Think of Delta as a smart folder of files.

Without delta it just a plain Parquet.
delta folder has _delta_log file. it is = brain of Delta
It stores JSON files describing: Which files exist, Which were removed ,Schema Versions


it's ACID Transactions. Either ALL data for a batch is written OR nothing is written No garbage.




In [0]:
query = flattened_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .start(output_path)
query.awaitTermination()

%md
we can't use the show() for stream stuffs.

#### Diplay() or show() doesn't work in streaming dataframe

use query to view the data. As done Below

In [0]:


%sql
SELECT * FROM delta.`/Volumes/pyspark/stream/sparkstreaming/output_table`

In [0]:
dbutils.fs.ls('/Volumes/pyspark/stream/sparkstreaming/sparksource')

In [0]:
dbutils.fs.ls('/Volumes/pyspark/stream/sparkstreaming/output_table')

In [0]:
#Read a sample of the source data to verify its structure and contents.

sample_df = spark.read.option('multiLine', True).json('/Volumes/pyspark/stream/sparkstreaming/sparksource')
display(sample_df.head(5))

In [0]:
#Count rows in the Delta output table to confirm if any data was written.

df = spark.read.format('delta').load('/Volumes/pyspark/stream/sparkstreaming/output_table')
display(df.count())

1. Output Format (delta, parquet, or console)
2. How to write (append, complete, or update)
3. Memory of where the stream left off
4. Where the data goes

In PySpark DataFrame API, we don’t really write SQL-style subqueries. Instead, we turn the subquery into a DataFrame and then join it. Conceptually:

SQL subquery → intermediate DataFrame → join